<a href="https://colab.research.google.com/github/priyal6/General-llm/blob/main/speculative_decoding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"

target_name="gpt2"
draft_name="distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(target_name)
target_model = AutoModelForCausalLM.from_pretrained(target_name).to(device).eval()
draft_model = AutoModelForCausalLM.from_pretrained(draft_name).to(device).eval()

prompt = "Speculative decoding is useful because"

input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
@torch.no_grad()
def greedy_next_token(model, input_ids):
  outputs = model(input_ids)
  logits = outputs.logits[:,-1,:]
  next_token = torch.argmax(logits, dim=-1, keepdim=True)
  return next_token

@torch.no_grad()
def draft_k_tokens(draft_model, input_ids, k=4):
  drafted = []
  current = input_ids

  for _ in range(k):
    next_token = greedy_next_token(draft_model, current)
    drafted.append(next_token)
    current = torch.cat([current, next_token], dim=1)

  return torch.cat(drafted, dim=1)

@torch.no_grad()
def speculative_decode(
    target_model,
    draft_model,
    input_ids,
    max_new_tokens=40,
    draft_k=4
):
  generated = input_ids.clone()
  total_drafted = 0
  total_accepted = 0

  while generated.shape[1] < input_ids.shape[1] + max_new_tokens:
    draft_tokens = draft_k_tokens(draft_model, generated, k=draft_k)
    total_drafted += draft_tokens.shape[1]
    accepted_tokens = []

    for i in range(draft_tokens.shape[1]):
      prefix = torch.cat(
          [generated] + accepted_tokens,
          dim=1
      ) if accepted_tokens else generated

      target_next = greedy_next_token(target_model, prefix)
      draft_next = draft_tokens[:, i]
      if torch.equal(target_next, draft_next):
        accepted_tokens.append(draft_next)
        total_accepted +=1
      else:
        accepted_tokens.append(target_next)
        break

    if accepted_tokens:
        generated = torch.cat([generated] + accepted_tokens, dim=1)
  acceptance_rate = total_accepted / total_drafted
  return generated, acceptance_rate

In [10]:
output_id, acceptance_rate = speculative_decode(
    target_model=target_model,
    draft_model=draft_model,
    input_ids=input_ids,
    max_new_tokens=50,
    draft_k=4
)

print(tokenizer.decode(output_id[0], skip_special_tokens=True))
print("Acceptance rate:", round(acceptance_rate, 3))

Speculative decoding is useful because it allows us to determine the exact meaning of a word. For example, if we want to know if a word is a noun or a verb, we can use the following:

"I am a man"

"I am a
Acceptance rate: 0.0
